In [ ]:
# Cell 1 — Install
!pip install yfinance -q

In [ ]:
# Cell 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os
OUTPUT_DIR = '/content/drive/MyDrive/ethical-finance/ohlcv_backfill'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Drive monté —', OUTPUT_DIR)

In [ ]:
# Cell 3 — Config
from datetime import date
START = '2026-05-15'
END = str(date.today())
print(f'Backfill {START} → {END}')

In [ ]:
# Cell 4 — Tickers SP500 + CAC40 + ETF
TICKERS = [
    # SP500 top 50
    'AAPL','MSFT','NVDA','AMZN','GOOGL','GOOG','META','TSLA','BRK-B','AVGO',
    'JPM','LLY','V','MA','UNH','XOM','PG','COST','HD','JNJ',
    'ABBV','BAC','MRK','CRM','CVX','NFLX','ORCL','AMD','ADBE','KO',
    'PEP','TMO','WMT','ACN','MCD','CSCO','ABT','GE','NOW','DHR',
    'ISRG','IBM','GS','AXP','SPGI','BKNG','TXN','QCOM','AMAT','INTU',
    # CAC40
    'AI.PA','AIR.PA','ALO.PA','ATO.PA','BN.PA','BNP.PA','CA.PA','CAP.PA',
    'CS.PA','DG.PA','DSY.PA','EL.PA','EN.PA','ENGI.PA','ERF.PA','GLE.PA',
    'HO.PA','KER.PA','LR.PA','MC.PA','ML.PA','MT.AS','OR.PA','ORA.PA',
    'PUB.PA','RI.PA','RMS.PA','RNO.PA','SAF.PA','SAN.PA','SGO.PA','STLAM.MI',
    'STM.PA','SW.PA','TEP.PA','TTE.PA','URW.PA','VIE.PA','VIV.PA','WLN.PA',
    # ETF
    'GLD','SLV','GDX','PDBC','IAU','PPLT',
    'SPY','QQQ','IWM','VTI','EFA','VEA',
    # Indices
    '^GSPC','^FCHI','^GDAXI','^VIX',
]
print(f'{len(TICKERS)} tickers')

In [ ]:
# Cell 5 — Download + save CSV par batch
import yfinance as yf
import pandas as pd
import time

all_dfs = []
errors = []

for i, ticker in enumerate(TICKERS):
    try:
        df = yf.download(ticker, start=START, end=END, progress=False, auto_adjust=True)
        if df.empty:
            errors.append(ticker)
            continue
        df = df.reset_index()
        df['ticker'] = ticker
        df = df.rename(columns={'Date':'date','Open':'open','High':'high','Low':'low','Close':'close','Volume':'volume'})
        df['adj_close'] = df['close']
        df = df[['ticker','date','open','high','low','close','adj_close','volume']]
        all_dfs.append(df)
        if i % 20 == 0:
            print(f'{i}/{len(TICKERS)} — {ticker} — {len(df)} rows')
    except Exception as e:
        errors.append(f'{ticker}: {e}')
    time.sleep(0.3)

result = pd.concat(all_dfs, ignore_index=True)
print(f'Total: {len(result)} rows — {result.ticker.nunique()} tickers')
print(f'Errors: {errors}')

In [ ]:
# Cell 6 — Save to Drive
import datetime
filename = f'ohlcv_backfill_{datetime.date.today()}.csv'
path = f'{OUTPUT_DIR}/{filename}'
result.to_csv(path, index=False)
print(f'Saved: {path}')
print(result.head())